# 🎯 Netelpro + RAFT: Entrenando Gemma 4 E2B a escribir programas Netelpro correctos
### *(Segundo modelo del experimento -- mismo curriculo, mismo verificador, linaje de entrenamiento distinto a Qwen)*

Rejection-sampling fine-tuning (RAFT/STaR): la recompensa es "compilo + paso los tests", no preferencia humana.

**Por que un segundo modelo:** un solo modelo (Qwen) no prueba que el metodo generaliza -- solo prueba que
funciona en Qwen. Gemma es de otro laboratorio (Google DeepMind), otro tokenizer, otro corpus de
entrenamiento. Mismos hiperparametros, mismo curriculo, mismo verificador -- lo UNICO que cambia es el
modelo base, a proposito (comparacion controlada).

**Por que este checkpoint especifico** (`google/gemma-4-E2B-it-qat-q4_0-unquantized`, no el
`unsloth/gemma-4-E2B` plano): es QAT (quantization-aware training) de Google -- entrenado CON ruido de
cuantizacion simulado, publicado en precision completa (SafeTensors) para poder seguir entrenandolo. Esta
pipeline termina exportando a GGUF -- arrancar de un checkpoint ya robusto a esa cuantizacion final tiene
sentido tecnico real.

**Correccion post-intento-1 (ver historial):** E2B/E4B son multimodales de verdad (texto+imagen+audio),
no solo texto. El primer intento uso `FastLanguageModel` (el loader de solo-texto) y fallo: reparto forzado
en 2 GPUs, capas de atencion K/V "MISSING" (inicializadas al azar por el loader equivocado), warning de
`target_modules` mal mapeados. Esta version usa `FastVisionModel` (el loader correcto para arquitecturas
multimodales, aunque afinemos solo texto) con `finetune_vision_layers=False` -- entrena solo lenguaje, sin
tocar la rama de imagen/audio, y fija la GPU a una sola tarjeta para evitar el reparto automatico que
complico el intento anterior.

Ver `docs/superpowers/specs/2026-09-07-rlvr-netelpro-raft-design.md` para el diseno completo del experimento.

---

## ⚠️ Antes de correr esto por primera vez en Kaggle (lee esto, no lo saltes)

1. **Verificacion de telefono, obligatoria.** Kaggle exige verificar tu numero por SMS para activar GPU/Internet. Perfil (arriba a la derecha) -> Settings -> Phone verification. **Usa un numero real de carrier, no VoIP.**
2. **Accelerator: GPU T4 x2** (Kaggle te da 2 tarjetas T4 aunque solo usemos una a proposito -- ver celda de configuracion de GPU mas abajo).
3. **Internet: On.** Sin esto, `pip install` y `git clone` fallan.
4. **Cuota:** ~30h/semana de GPU T4, se resetea cada semana. Sesion individual tope 12h.
5. **VRAM:** segun la guia oficial de Unsloth, E2B con LoRA entra en 8-10GB -- una sola T4 (14.5GB efectivos en Kaggle) alcanza de sobra sin repartir en 2 GPUs.
6. **Para que el resultado sobreviva el cierre de sesion:** "Save Version" -> "Save & Run All (Commit)" en vez de solo correr interactivo.


## 1. Instalación de dependencias

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"  # Kaggle a veces pide una API key de wandb a mitad del entrenamiento y cuelga el kernel si no se desactiva antes.

# Unsloth + TRL para fine-tune LoRA rapido en T4, llvmlite para netelpro.
# unsloth[kaggle-new] (no colab-new): el extra correcto para la imagen base de Kaggle.
!pip install --no-deps "xformers<0.0.29" "trl<0.15.0" peft accelerate bitsandbytes triton
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q llvmlite>=0.49


### 1b. Verificar GPU (antes de importar unsloth)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle da 2 T4s; E2B entra comodo en una sola (8-10GB segun Unsloth). Evita el reparto automatico entre 2 GPUs que rompio el intento anterior.

# Unsloth se niega a importar sin acelerador CUDA, con un NotImplementedError criptico.
# Fallamos temprano y en claro: sin GPU no hay experimento.
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU activa. En Kaggle: panel derecho -> Settings -> Accelerator -> GPU T4 x2 "
        "(requiere telefono verificado, ver celda de arriba) -> el kernel se reinicia: corre todo de nuevo desde la celda 1."
    )
print("GPU OK:", torch.cuda.get_device_name(0))


## 2. Clonar el repo y cargar el corpus de tareas RLVR

In [ ]:
# Requiere 'Internet: On' en Settings (panel derecho) -- sin eso este git clone y los pip install de arriba fallan.
# Clonamos el repo para tener netelpro/ y rlvr/ disponibles como paquetes
!git clone https://github.com/jona2428/netelpro.git

import sys
sys.path.insert(0, "netelpro")

from rlvr.tasks import load_all_tasks, split_train_ood
from rlvr.prompting import build_prompt
from rlvr.verify import verify_program

all_tasks = load_all_tasks()
train_ids, ood_ids = split_train_ood(list(all_tasks.keys()), ood_fraction=0.2)
from rlvr.tasks import OOD_TASK_IDS
assert sorted(ood_ids) == sorted(OOD_TASK_IDS), "held-out explicito: split debe calzar con OOD_TASK_IDS"
print(f"Corpus: {len(all_tasks)} tareas -- {len(train_ids)} train, {len(ood_ids)} OOD (held-out)")


## 3. Configuración del experimento

In [ ]:
NUM_ROUNDS = 5  # v2: 3->5 -- el plateau de la v1 en ronda 2 sugiere falta de señal
SAMPLES_PER_TASK = 16  # v2: 8->16 -- más candidatos por tarea, más señal de SFT por ronda
MAX_KEEP_PER_TASK = 2  # tope de candidatos que pasan por tarea, evita desbalancear el SFT
MAX_NEW_TOKENS = 256
SAMPLING_TEMPERATURE = 0.8
NUM_TEST_CASES = 20
EVAL_SEED = 0
MAX_VERIFY_STEPS = 1_000_000  # presupuesto de pasos del verificador (fix C1: runaway tail-recursion no cuelga Colab)
MODEL_NAME = "google/gemma-4-E2B-it-qat-q4_0-unquantized"  # QAT de Google, SafeTensors entrenable -- NO el checkpoint RAFT de Qwen ya entrenado


## 4. Cargar el modelo base + LoRA (multimodal -- FastVisionModel)
E2B/E4B son multimodales (texto+imagen+audio) aunque afinemos solo texto. `FastLanguageModel` (el loader
de solo-texto que usa `train_colab.ipynb` para Qwen) NO sirve aca -- produce capas de atencion
inicializadas al azar y fuerza un reparto innecesario en 2 GPUs. `finetune_vision_layers=False` entrena
solo la rama de lenguaje, sin tocar imagen/audio.


In [ ]:
from unsloth import FastVisionModel
import torch

max_seq_length = 1024

model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# El resto del notebook usa la variable "tokenizer" (apply_chat_template,
# sample_completions, etc.) -- alias para no tener que tocar esas celdas.
tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # RAFT es solo texto (codigo Netelpro) -- no tocar imagen/audio
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=42,
    target_modules="all-linear",       # no lista manual -- Unsloth resuelve el mapeo correcto en arquitecturas multimodales
)


## 5. Muestreo y extracción de código Netelpro

In [ ]:
def extract_sl_code(raw_text: str) -> str:
    """El modelo puede envolver el programa en un bloque de código markdown --
    si hay un fence ``` lo extraemos, si no devolvemos el texto tal cual."""
    if "```" in raw_text:
        parts = raw_text.split("```")
        if len(parts) >= 2:
            candidate = parts[1]
            candidate = candidate.removeprefix("netelpro").removeprefix("lisp").strip()
            return candidate
    return raw_text.strip()


def sample_completions(prompt: str, n: int, temperature: float) -> list[str]:
    FastVisionModel.for_inference(model)
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([chat] * n, return_tensors="pt", padding=True).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=temperature,
    )
    texts = tokenizer.batch_decode(
        outputs[:, inputs.input_ids.shape[1] :], skip_special_tokens=True
    )
    return [extract_sl_code(t) for t in texts]


## 6. Pass rate en el split OOD (criterio de éxito, spec §7)

In [ ]:
def evaluate_pass_rate(task_ids: list[str], num_samples: int) -> tuple[float, list[str]]:
    # task_id ES la clave de all_tasks (load_all_tasks() carga por
    # nombre de módulo, y por construcción TASK_ID == nombre de módulo
    # en todo el corpus -- ver rlvr/tasks/*.py). Sin indirección.
    # Seed fija antes de muestrear (v2): baseline, rondas y eval final comparten
    # los draws del sampler -> comparación pareada (caveat de la v1 cerrado).
    torch.manual_seed(EVAL_SEED)
    passed_ids: list[str] = []
    for task_id in task_ids:
        task_module = all_tasks[task_id]
        prompt = build_prompt(task_module)
        candidates = sample_completions(prompt, num_samples, SAMPLING_TEMPERATURE)
        if any(
            verify_program(
                c, task_module, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS
            ).passed
            for c in candidates
        ):
            passed_ids.append(task_id)
    # v3: devuelve también QUÉ tareas pasaron, no solo el %, para el reporte
    # con detalle por tarea (celda 10) -- antes esa info se descartaba.
    return len(passed_ids) / len(task_ids), passed_ids


baseline_pass_rate, baseline_passed_ids = evaluate_pass_rate(ood_ids, num_samples=SAMPLES_PER_TASK)
print(f"[baseline, sin entrenar] pass rate OOD: {baseline_pass_rate:.1%}")
print(f"[baseline] tareas resueltas: {baseline_passed_ids}")


## 7. Loop RAFT iterativo**v2 -- pool acumulado:** cada ronda SFT entrena sobre TODO el harvest verificado hasta ahora (no solo el fresco) -- RAFT canónico; evita el forgetting que planchó la ronda 2 de la v1.

In [ ]:
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

all_sft_examples: list[dict] = []  # v2: pool acumulado -- RAFT canónico
round_history: list[dict] = []  # v3: trayectoria completa para el reporte (celda 10)
for round_num in range(NUM_ROUNDS):
    print(f"\n=== Ronda {round_num} ===")
    sft_examples = []
    for task_id in train_ids:
        task_module = all_tasks[task_id]
        prompt = build_prompt(task_module)
        candidates = sample_completions(prompt, SAMPLES_PER_TASK, SAMPLING_TEMPERATURE)
        kept = 0
        for candidate in candidates:
            if kept >= MAX_KEEP_PER_TASK:
                break
            result = verify_program(candidate, task_module, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS)
            if result.passed:
                sft_examples.append({"prompt": prompt, "completion": candidate})
                kept += 1
    all_sft_examples.extend(sft_examples)  # v2: acumular, nunca descartar lo verificado
    print(f"Ronda {round_num}: {len(sft_examples)} nuevos -- pool acumulado: {len(all_sft_examples)}")

    if not sft_examples:
        print("Ninguna tarea de train produjo un candidato que pasara -- se aborta esta ronda")
        continue

    round_dataset = Dataset.from_list(all_sft_examples)  # v2: entrena sobre el pool completo
    FastVisionModel.for_training(model)

    def format_sft_example(example):
        # Combina prompt + completion en UNA sola secuencia de entrenamiento,
        # usando el mismo apply_chat_template que sample_completions (celda 5) --
        # si el formato de entrenamiento diverge del de muestreo, el modelo
        # entrena en un formato y samplea en otro.
        chat = tokenizer.apply_chat_template(
            [{"role": "user", "content": example["prompt"]}],
            tokenize=False,
            add_generation_prompt=True,
        )
        return chat + example["completion"]

    sft_args = SFTConfig(
        output_dir=f"netelpro_raft_round_{round_num}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        logging_steps=1,  # v2: rondas cortas -- con 5 la tabla de loss salía vacía
        save_strategy="no",
        warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        # Dataset rows carry prompt/completion keys, so TRL classifies it as
        # prompt-completion and defaults completion_only_loss=True -- which
        # Unsloth's fork rejects alongside formatting_func. Explicit False
        # selects full-sequence loss (prompt + completion), the canonical
        # RAFT objective, and unblocks the formatter path.
        completion_only_loss=False,
    )
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=round_dataset,
        formatting_func=format_sft_example,
    )
    sft_trainer.train()

    round_pass_rate, round_passed_ids = evaluate_pass_rate(ood_ids, num_samples=SAMPLES_PER_TASK)
    round_history.append({
        "round": round_num,
        "pass_rate": round_pass_rate,
        "passed_ids": round_passed_ids,
        "pool_size": len(all_sft_examples),
    })
    print(f"[ronda {round_num}] pass rate OOD: {round_pass_rate:.1%} -- resueltas: {round_passed_ids}")


## 8. Criterio de éxito (spec §7)
El RAFT tiene que superar CLARAMENTE al baseline sin entrenar -- si no, no está enseñando nada que el prompt no diera gratis.

In [ ]:
final_pass_rate, final_passed_ids = evaluate_pass_rate(ood_ids, num_samples=SAMPLES_PER_TASK)
print(f"Baseline (sin entrenar): {baseline_pass_rate:.1%}")
print(f"Final (tras {NUM_ROUNDS} rondas de RAFT): {final_pass_rate:.1%}")
if final_pass_rate > baseline_pass_rate + 0.1:
    print("RAFT superó claramente al baseline -- la señal de recompensa enseñó algo real.")
else:
    print("RAFT NO superó claramente al baseline -- no declarar éxito, reportar el número tal cual.")


## 9. Exportar a GGUF

In [ ]:
model.save_pretrained_gguf(
    "netelpro_qwen1.5b_raft", tokenizer, quantization_method="q4_k_m"
)
print("✅ Modelo GGUF exportado en la carpeta 'netelpro_qwen1.5b_raft'.")


## 10. Reporte final (trayectoria + detalle por tarea)

In [ ]:
import datetime

report_lines = []
now = datetime.datetime.now(datetime.timezone.utc).isoformat()
report_lines.append(f"# Netelpro RAFT -- reporte de corrida ({now})")
report_lines.append("")
report_lines.append(f"- Corpus: {len(all_tasks)} tareas -- {len(train_ids)} train, {len(ood_ids)} OOD")
report_lines.append(
    f"- Rondas: {NUM_ROUNDS}, muestras/tarea: {SAMPLES_PER_TASK}, "
    f"pool SFT final: {len(all_sft_examples)} ejemplos"
)
report_lines.append("")
report_lines.append(f"## Trayectoria OOD (pass@{SAMPLES_PER_TASK}, seed={EVAL_SEED})")
report_lines.append("")
report_lines.append("| Etapa | Pass rate | Tareas resueltas |")
report_lines.append("|---|---|---|")
report_lines.append(
    f"| Baseline | {baseline_pass_rate:.1%} | {len(baseline_passed_ids)}/{len(ood_ids)} |"
)
for entry in round_history:
    report_lines.append(
        f"| Ronda {entry['round']} | {entry['pass_rate']:.1%} | "
        f"{len(entry['passed_ids'])}/{len(ood_ids)} |"
    )
report_lines.append(
    f"| Final | {final_pass_rate:.1%} | {len(final_passed_ids)}/{len(ood_ids)} |"
)
report_lines.append("")
report_lines.append("## Detalle por tarea OOD (baseline vs final)")
report_lines.append("")
report_lines.append("| Tarea | Baseline | Final |")
report_lines.append("|---|---|---|")
for tid in sorted(ood_ids):
    b = "si" if tid in baseline_passed_ids else "no"
    f = "si" if tid in final_passed_ids else "no"
    marker = " <-- curriculum gcd (run #3)" if tid == "gcd_pair" else ""
    report_lines.append(f"| {tid}{marker} | {b} | {f} |")
report_lines.append("")
if final_pass_rate > baseline_pass_rate + 0.1:
    report_lines.append("**Veredicto:** RAFT superó claramente al baseline.")
else:
    report_lines.append(
        "**Veredicto:** RAFT NO superó claramente al baseline -- reportado tal cual, "
        "sin declarar éxito."
    )

report_text = "\n".join(report_lines)
with open("informe_run.md", "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)


## 11. Descargar resultados (GGUF + reporte)

In [ ]:
# Kaggle no tiene google.colab.files -- el output vive en /kaggle/working/ (cwd
# por defecto en Kaggle notebooks) y sobrevive al cierre de sesion SOLO si
# corriste "Save Version -> Save & Run All (Commit)" (ver aviso al inicio del notebook).
# FileLink da un enlace clickeable para bajar manualmente ademas, por si acaso.
import shutil
from IPython.display import FileLink, display

shutil.make_archive("netelpro_qwen1.5b_raft", "zip", "netelpro_qwen1.5b_raft")
display(FileLink("netelpro_qwen1.5b_raft.zip"))
display(FileLink("informe_run.md"))


## 12. Sweep de cuantizaciones -- comparar pass@8 OOD en TODOS los niveles GGUF disponibles

Un solo punto de dato (q4_k_m) no dice si perder precision cuesta algo real. Exportamos a cada nivel
que expone `save_pretrained_gguf` (la familia i-quant completa de llama.cpp NO esta disponible por esta
via -- solo `iq2_xxs`, `iq2_xs`, `iq3_xxs` -- variantes como `iq4_nl`/`iq1_s` requeririan `llama-quantize`
de llama.cpp directo sobre el f16, no esta funcion) y evaluamos cada uno con el mismo verificador y el
mismo split OOD que ya se uso arriba.

**Costo:** 9 niveles x 20 tareas x `SAMPLES_PER_TASK` muestras -- con el default (16) son 2880
generaciones. Si te preocupa el limite de sesion de Kaggle (12h), baja `SWEEP_SAMPLES_PER_TASK` en la
celda de abajo antes de correr.


In [ ]:
# Libera la VRAM del modelo de entrenamiento antes del sweep -- si se deja
# cargado junto con varios GGUF de llama.cpp, una sola T4 no alcanza.
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print("VRAM en uso tras liberar:", torch.cuda.memory_allocated() / 1e9, "GB")


### 12a. Exportar a cada nivel de cuantizacion


In [ ]:
import os

SWEEP_SAMPLES_PER_TASK = SAMPLES_PER_TASK  # bajalo (ej. 4) si te preocupa el tiempo de sesion

QUANT_METHODS = [
    "q8_0",     # referencia casi sin perdida
    "q6_k",
    "q5_k_m",
    "q4_k_m",   # el que ya se uso en la corrida de Qwen -- punto de comparacion directo
    "q3_k_m",
    "q2_k",     # extremo agresivo de los k-quants
    "iq3_xxs",  # extremo agresivo de los i-quants disponibles via save_pretrained_gguf
    "iq2_xs",
    "iq2_xxs",
]

# Recarga el checkpoint final entrenado (guardado por la celda de exportacion
# GGUF original, mas arriba) -- separado del modelo que se libero recien.
from unsloth import FastVisionModel
model, processor = FastVisionModel.from_pretrained(
    "netelpro_qwen1.5b_raft",  # ajusta al nombre real de la carpeta donde se guardo el checkpoint final
    load_in_4bit=True,
)
tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

gguf_paths = {}
for method in QUANT_METHODS:
    out_dir = f"netelpro_gemma_{method}"
    print(f"\n=== Exportando {method} ===")
    model.save_pretrained_gguf(out_dir, tokenizer, quantization_method=method)
    gguf_file = next((os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.endswith(".gguf")), None)
    gguf_paths[method] = gguf_file
    size_mb = os.path.getsize(gguf_file) / 1e6 if gguf_file else 0
    print(f"{method}: {gguf_file} ({size_mb:.0f} MB)")


### 12b. Instalar llama-cpp-python con CUDA (compila desde fuente, tarda unos minutos)


In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q "llama-cpp-python[server]" --no-cache-dir


### 12c. Evaluar pass@8 OOD en cada nivel de cuantizacion
Reusa `build_prompt`, `verify_program`, `ood_ids` de las celdas de arriba. El prompt se formatea con el
`tokenizer` (no dejamos que llama.cpp adivine su propio template) para que sea identico al de la corrida fp16.


In [ ]:
import gc
import time
from llama_cpp import Llama

results = []

for method, gguf_path in gguf_paths.items():
    print(f"\n=== Evaluando {method} ===")
    llm = Llama(model_path=gguf_path, n_gpu_layers=-1, n_ctx=1024, verbose=False)

    passed_ids = []
    total_tokens = 0
    total_time = 0.0

    for task_id in ood_ids:
        task_module = all_tasks[task_id]
        prompt = build_prompt(task_module)
        chat = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
        )

        candidates = []
        for _ in range(SWEEP_SAMPLES_PER_TASK):
            t0 = time.time()
            out = llm(chat, max_tokens=MAX_NEW_TOKENS, temperature=SAMPLING_TEMPERATURE)
            total_time += time.time() - t0
            text = out["choices"][0]["text"]
            total_tokens += out["usage"]["completion_tokens"]
            candidates.append(extract_sl_code(text))

        if any(
            verify_program(c, task_module, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS).passed
            for c in candidates
        ):
            passed_ids.append(task_id)

    pass_rate = len(passed_ids) / len(ood_ids)
    tok_per_sec = total_tokens / total_time if total_time > 0 else 0
    size_mb = os.path.getsize(gguf_path) / 1e6

    results.append({
        "quant": method,
        "size_mb": round(size_mb, 1),
        "pass_rate": pass_rate,
        "passed": passed_ids,
        "tokens_per_sec": round(tok_per_sec, 1),
    })
    print(f"{method}: pass@{SWEEP_SAMPLES_PER_TASK} OOD = {pass_rate:.1%}  |  {size_mb:.0f} MB  |  {tok_per_sec:.1f} tok/s")

    del llm
    gc.collect()


### 12d. Tabla final


In [ ]:
print(f"{'quant':<10} {'MB':>7} {'pass@8':>8} {'tok/s':>8}")
for r in sorted(results, key=lambda r: r["size_mb"], reverse=True):
    print(f"{r['quant']:<10} {r['size_mb']:>7.0f} {r['pass_rate']:>7.1%} {r['tokens_per_sec']:>7.1f}")
